In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
import pandas as pd
dataset= pd.read_csv("/content/drive/MyDrive/AI_ML_Foundation/datasets/cleaned_titanic_dataset.csv")

dataset.head()

,Unnamed: 0,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,TItle
0,0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1.0,0.0,A/5 21171,7.2500,Unknown Cabin,S,Mr,Mr
1,1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1.0,0.0,PC 17599,71.2833,C85,C,Mrs,Mrs
2,2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0.0,0.0,STON/O2. 3101282,7.9250,Unknown Cabin,S,Miss,Miss
3,3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1.0,0.0,113803,53.1000,C123,S,Mrs,Mrs
4,4,5,0,3,"Allen, Mr. William Henry",male,35.0,0.0,0.0,373450,8.0500,Unknown Cabin,S,Mr,Mr


In [7]:
# statistics.

### Hypothesis Testing Example: Survival Rate by Gender

Let's formulate a hypothesis and test it using the Titanic dataset. A common question is whether there's a significant difference in survival rates between male and female passengers.

**Null Hypothesis ($H_0$):** There is no significant difference in the mean survival rate between male and female passengers.

**Alternative Hypothesis ($H_1$):** There is a significant difference in the mean survival rate between male and female passengers.

We will use a two-sample independent t-test, which is appropriate for comparing the means of two independent groups (male and female survival rates).

In [8]:
from scipy import stats
import numpy as np

# Separate survival rates for male and female
survival_male = dataset[dataset['Sex'] == 'male']['Survived']
survival_female = dataset[dataset['Sex'] == 'female']['Survived']

# Perform independent two-sample t-test
t_statistic, p_value = stats.ttest_ind(survival_male, survival_female, equal_var=False) # Welch's t-test assuming unequal variances

print(f"T-statistic: {t_statistic:.3f}")
print(f"P-value: {p_value:.3f}")

T-statistic: -18.672
P-value: 0.000


### P-value Interpretation

The **p-value** tells us the probability of observing a test statistic as extreme as, or more extreme than, the one calculated, assuming the null hypothesis is true.

*   If $p < \alpha$ (our chosen significance level, commonly 0.05), we reject the null hypothesis.
*   If $p \geq \alpha$, we fail to reject the null hypothesis.

In our case, the p-value is quite small. This suggests that if there were truly no difference in survival rates between males and females, observing such a large difference in our sample would be very unlikely. Therefore, we would likely reject the null hypothesis.

In [10]:
# Calculate Cohen's d (Effect Size)
def cohen_d(x, y):
    nx = len(x)
    print(nx)
    ny = len(y)
    print(ny)
    dof = nx + ny - 2
    print(dof)
    pooled_std = np.sqrt(((nx - 1) * np.std(x, ddof=1)**2 + (ny - 1) * np.std(y, ddof=1)**2) / dof)
    print(pooled_std)
    return (np.mean(x) - np.mean(y)) / pooled_std

cohens_d_value = cohen_d(survival_male, survival_female)
print(f"Cohen's d (Effect Size): {cohens_d_value:.3f}")

577
314
889
0.40872666776971966
Cohen's d (Effect Size): -1.353


### Effect Size — Cohen's d Interpretation

**Cohen's d** measures the standardized difference between two means, indicating the magnitude of the effect. It helps us understand the practical significance of the difference, regardless of sample size.

*   **Small effect:** $|d| < 0.2$
*   **Medium effect:** $0.2 \leq |d| < 0.5$
*   **Large effect:** $|d| \geq 0.8$

Our calculated Cohen's d value indicates a very large effect size. This means the difference in survival rates between genders is substantial and not just statistically significant, but also practically meaningful.

In [11]:
# Calculate Confidence Interval for the difference in means
alpha = 0.05  # 95% confidence level
diff_means = np.mean(survival_male) - np.mean(survival_female)

# Standard error of the difference (using Welch-Satterthwaite equation for unequal variances)
sem_male = np.std(survival_male, ddof=1) / np.sqrt(len(survival_male))
sem_female = np.std(survival_female, ddof=1) / np.sqrt(len(survival_female))

std_err_diff = np.sqrt(sem_male**2 + sem_female**2)

# Degrees of freedom for Welch's t-test
df_welch = (sem_male**2 + sem_female**2)**2 / \
           ((sem_male**4 / (len(survival_male) - 1)) + (sem_female**4 / (len(survival_female) - 1)))

# Get the critical t-value
confidence_level = 1 - alpha
t_critical = stats.t.ppf(1 - alpha/2, df_welch)

margin_of_error = t_critical * std_err_diff

confidence_interval_lower = diff_means - margin_of_error
confidence_interval_upper = diff_means + margin_of_error

print(f"Difference in means (Male - Female): {diff_means:.3f}")
print(f"95% Confidence Interval for the difference: ({confidence_interval_lower:.3f}, {confidence_interval_upper:.3f})")

Difference in means (Male - Female): -0.553
95% Confidence Interval for the difference: (-0.611, -0.495)


### Confidence Interval Interpretation

A **confidence interval** provides a range of values within which the true population parameter (in this case, the true difference in mean survival rates between male and female passengers) is likely to lie, with a certain level of confidence (e.g., 95%).

*   It is not about the probability that the true value is *in* the range for this specific interval, but rather that if you were to repeat the experiment many times, 95% of the intervals constructed would contain the true population parameter.

Our confidence interval for the difference in survival rates (Male - Female) does not include zero. This further supports the conclusion that there is a statistically significant difference between the survival rates of males and females on the Titanic. The interval shows that the mean survival rate for males is substantially lower than for females.

### Summary of Findings

Based on our analysis:

1.  **P-value:** The very small p-value (much less than 0.05) leads us to **reject the null hypothesis**. This indicates strong evidence that there is a statistically significant difference in survival rates between male and female passengers on the Titanic.
2.  **Effect Size (Cohen's d):** The large Cohen's d value shows that this difference is not only statistically significant but also a **large and practically meaningful effect**.
3.  **Confidence Interval:** The 95% confidence interval for the difference in means does not contain zero, further reinforcing the statistical significance and providing a plausible range for the true difference in population survival rates.

### Hypothesis Testing Example: Age by Survival Status

Let's test if there's a significant difference in the average age of passengers who survived compared to those who did not.

**Null Hypothesis ($H_0$):** There is no significant difference in the mean age between passengers who survived and those who did not.

**Alternative Hypothesis ($H_1$):** There is a significant difference in the mean age between passengers who survived and those who did not.

In [14]:
from scipy import stats
import numpy as np

# Drop rows where 'Age' is NaN, as t-test cannot handle missing values
dataset_age_clean = dataset.dropna(subset=['Age'])

# Separate age for survived and not survived passengers
age_survived = dataset_age_clean[dataset_age_clean['Survived'] == 1]['Age']
age_not_survived = dataset_age_clean[dataset_age_clean['Survived'] == 0]['Age']

# Perform independent two-sample t-test (Welch's t-test for unequal variances)
t_statistic_age, p_value_age = stats.ttest_ind(age_survived, age_not_survived, equal_var=False)

print(f"T-statistic (Age by Survival): {t_statistic_age:.3f}")
print(f"P-value (Age by Survival): {p_value_age:.3f}")

T-statistic (Age by Survival): -0.879
P-value (Age by Survival): 0.380


### P-value Interpretation for Age and Survival

Based on the calculated p-value, we can interpret the statistical significance of the difference in ages between the two survival groups.

If the p-value is less than our significance level (e.g., 0.05), we reject the null hypothesis, suggesting a statistically significant difference in age between those who survived and those who did not. Otherwise, we fail to reject the null hypothesis.

In [16]:
# Calculate Cohen's d (Effect Size) for Age and Survival
def cohen_d(x, y):
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((nx - 1) * np.std(x, ddof=1)**2 + (ny - 1) * np.std(y, ddof=1)**2) / dof)
    return (np.mean(x) - np.mean(y)) / pooled_std

cohens_d_age_survival = cohen_d(age_survived, age_not_survived)
print(f"Cohen's d (Effect Size for Age and Survival): {cohens_d_age_survival:.3f}")

Cohen's d (Effect Size for Age and Survival): -0.062


### Effect Size — Cohen's d Interpretation for Age and Survival

This Cohen's d value indicates the magnitude of the difference in average age between survivors and non-survivors. You can use the same interpretation guidelines:

*   **Small effect:** $|d| < 0.2$
*   **Medium effect:** $0.2 \leq |d| < 0.5$
*   **Large effect:** $|d| \geq 0.8$

In [17]:
# Calculate Confidence Interval for the difference in means (Age Survived - Age Not Survived)
alpha = 0.05  # 95% confidence level
diff_means_age = np.mean(age_survived) - np.mean(age_not_survived)

# Standard error of the difference (using Welch-Satterthwaite equation for unequal variances)
sem_survived = np.std(age_survived, ddof=1) / np.sqrt(len(age_survived))
sem_not_survived = np.std(age_not_survived, ddof=1) / np.sqrt(len(age_not_survived))

std_err_diff_age = np.sqrt(sem_survived**2 + sem_not_survived**2)

# Degrees of freedom for Welch's t-test
df_welch_age = (sem_survived**2 + sem_not_survived**2)**2 / \
               ((sem_survived**4 / (len(age_survived) - 1)) + (sem_not_survived**4 / (len(age_not_survived) - 1)))

# Get the critical t-value
t_critical_age = stats.t.ppf(1 - alpha/2, df_welch_age)

margin_of_error_age = t_critical_age * std_err_diff_age

confidence_interval_lower_age = diff_means_age - margin_of_error_age
confidence_interval_upper_age = diff_means_age + margin_of_error_age

print(f"Difference in means (Age Survived - Age Not Survived): {diff_means_age:.3f}")
print(f"95% Confidence Interval for the difference (Age): ({confidence_interval_lower_age:.3f}, {confidence_interval_upper_age:.3f})")

Difference in means (Age Survived - Age Not Survived): -0.767
95% Confidence Interval for the difference (Age): (-2.482, 0.948)


### Confidence Interval Interpretation for Age and Survival

This confidence interval provides a range for the true difference in mean age between the population of survivors and non-survivors. If the interval does not include zero, it further supports the finding of a statistically significant difference.

### Hypothesis Testing Example: Fare by Survival Status

Let's investigate if the average fare paid differs significantly between passengers who survived and those who did not.

**Null Hypothesis ($H_0$):** There is no significant difference in the mean fare paid by passengers who survived and those who did not.

**Alternative Hypothesis ($H_1$):** There is a significant difference in the mean fare paid by passengers who survived and those who did not.

In [18]:
from scipy import stats
import numpy as np

# Separate fare for survived and not survived passengers
fare_survived = dataset[dataset['Survived'] == 1]['Fare']
fare_not_survived = dataset[dataset['Survived'] == 0]['Fare']

# Perform independent two-sample t-test (Welch's t-test for unequal variances)
t_statistic_fare, p_value_fare = stats.ttest_ind(fare_survived, fare_not_survived, equal_var=False)

print(f"T-statistic (Fare by Survival): {t_statistic_fare:.3f}")
print(f"P-value (Fare by Survival): {p_value_fare:.3f}")

T-statistic (Fare by Survival): 8.446
P-value (Fare by Survival): 0.000


### P-value Interpretation for Fare and Survival

Just like with the previous tests, the p-value here indicates the probability of observing our results (or more extreme) if there were truly no difference in mean fare between survivors and non-survivors. A small p-value would suggest a statistically significant difference.

In [19]:
# Calculate Cohen's d (Effect Size) for Fare and Survival
# Re-using the previously defined cohen_d function

cohens_d_fare_survival = cohen_d(fare_survived, fare_not_survived)
print(f"Cohen's d (Effect Size for Fare and Survival): {cohens_d_fare_survival:.3f}")

Cohen's d (Effect Size for Fare and Survival): 0.655


### Effect Size — Cohen's d Interpretation for Fare and Survival

This Cohen's d value quantifies the magnitude of the difference in average fare between those who survived and those who did not. Use the same interpretation rules for small, medium, and large effects.

In [ ]:
# Calculate Confidence Interval for the difference in means (Fare Survived - Fare Not Survived)
alpha = 0.05  # 95% confidence level
diff_means_fare = np.mean(fare_survived) - np.mean(fare_not_survived)

# Standard error of the difference (using Welch-Satterthwaite equation for unequal variances)
sem_fare_survived = np.std(fare_survived, ddof=1) / np.sqrt(len(fare_survived))
sem_fare_not_survived = np.std(fare_not_survived, ddof=1) / np.sqrt(len(fare_not_survived))

std_err_diff_fare = np.sqrt(sem_fare_survived**2 + sem_fare_not_survived**2)

# Degrees of freedom for Welch's t-test
df_welch_fare = (sem_fare_survived**2 + sem_fare_not_survived**2)**2 / \
                ((sem_fare_survived**4 / (len(fare_survived) - 1)) + (sem_fare_not_survived**4 / (len(fare_not_survived) - 1)))

# Get the critical t-value
t_critical_fare = stats.t.ppf(1 - alpha/2, df_welch_fare)

margin_of_error_fare = t_critical_fare * std_err_diff_fare

confidence_interval_lower_fare = diff_means_fare - margin_of_error_fare
confidence_interval_upper_fare = diff_means_fare + margin_of_error_fare

print(f"Difference in means (Fare Survived - Fare Not Survived): {diff_means_fare:.3f}")
print(f"95% Confidence Interval for the difference (Fare): ({confidence_interval_lower_fare:.3f}, {confidence_interval_upper_fare:.3f})")

### Confidence Interval Interpretation for Fare and Survival

This confidence interval will give you a range for the true difference in the mean fare between the population of survivors and non-survivors. Pay attention to whether the interval includes zero, as this helps in confirming the statistical significance.

In [ ]:
from scipy import